In [1]:
from transformers import Qwen3Model, AutoTokenizer
from google.colab import userdata
import torch
import os

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')



In [2]:
model_name = "Qwen/Qwen3-0.6B"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_name, dtype=torch.float32)

model = Qwen3Model.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    device_map = "auto",
)

model.to(device)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

[transformers] Qwen3Model LOAD REPORT from: Qwen/Qwen3-0.6B
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Qwen3Model(
  (embed_tokens): Embedding(151936, 1024)
  (layers): ModuleList(
    (0-27): 28 x Qwen3DecoderLayer(
      (self_attn): Qwen3Attention(
        (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
        (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
        (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
      )
      (mlp): Qwen3MLP(
        (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
        (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
        (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
      (post_attention_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
    )
  )
  (norm): Qwen3RM

In [3]:
model

Qwen3Model(
  (embed_tokens): Embedding(151936, 1024)
  (layers): ModuleList(
    (0-27): 28 x Qwen3DecoderLayer(
      (self_attn): Qwen3Attention(
        (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
        (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
        (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
      )
      (mlp): Qwen3MLP(
        (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
        (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
        (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
      (post_attention_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
    )
  )
  (norm): Qwen3RM



```
model(
     input_ids: torch.LongTensor | None = None,
      attention_mask: torch.Tensor | None = None,
      position_ids: torch.LongTensor | None = None,
      past_key_values: Cache | None = None,
      inputs_embeds: torch.FloatTensor | None = None,
      use_cache: bool | None = None,
      **kwargs: Unpack[TransformersKwargs],
)
```



In [4]:
sentence = "What is the capital of Spain?"
input_ids = tokenizer.encode(sentence,return_tensors="pt")
position_ids = torch.arange(input_ids.shape[1]).unsqueeze(0).expand(input_ids.shape)
attention_mask = torch.ones_like(input_ids)

input_ids = input_ids.to(device)
attention_mask = attention_mask.to(device)
position_ids = position_ids.to(device)

In [5]:
output = model(
    input_ids = input_ids,
    attention_mask=attention_mask,
    position_ids=position_ids,
    past_key_values=None,
    inputs_embeds=None,
    use_cache=False,
)

In [22]:
output.last_hidden_state.shape

torch.Size([1, 7, 1024])



```
"""
What is the capital of Spain?
<option>Valencia</option>
<option>Madrid</option>
<option>Barcelona</option>

Answer is :
"""

```



In [6]:
def create_input_tokens(question:str,choices:list) -> tuple[list,dict]:
  question_tokens = tokenizer.encode(question)

  choice_begin_token = tokenizer.encode("<option>")
  choice_end_token = tokenizer.encode("</option>")

  choices_tokens = [tokenizer.encode(choice) for choice in choices]

  answer_prompt = "Answer is:"
  answer_tokens = tokenizer.encode(answer_prompt)

  final_tokens = []
  final_tokens.extend(question_tokens)

  idxs = {
      "question_end_indx" : len(final_tokens) - 1,
      "choices_end_indxs" : []
  }

  for choice in choices_tokens:
      final_tokens.extend(choice_begin_token)
      final_tokens.extend(choice)
      final_tokens.extend(choice_end_token)
      idxs["choices_end_indxs"].append(len(final_tokens) - 1)

  final_tokens.extend(answer_tokens)
  idxs["answer_end_indx"] = len(final_tokens) - 1

  return final_tokens, idxs

In [7]:
question = "What is the capital of Spain?"
choices = ["Valencia","Madrid","Barcelona"]
input_ids,idxs = create_input_tokens(question,choices)
idxs

{'question_end_indx': 6,
 'choices_end_indxs': [13, 20, 27],
 'answer_end_indx': 30}

In [9]:
def generate_position_ids(idxs:dict) -> list:
  position_ids = [idx for idx in range(idxs["question_end_indx"] + 1)]

  start_idx = len(position_ids)
  choice_start_idx = start_idx


  for choice_end_idx in idxs["choices_end_indxs"]:
      position_ids.extend([idx for idx in range(start_idx ,start_idx + (choice_end_idx - choice_start_idx) + 1)])
      choice_start_idx = choice_end_idx + 1


  answer_len = (idxs['answer_end_indx'] + 1) - (idxs["choices_end_indxs"][-1] + 1)
  position_ids.extend([idx for idx in range(start_idx, start_idx + answer_len)])

  return position_ids

In [10]:
position_ids = generate_position_ids(idxs)
print(f"len position ids = {len(position_ids)} , len input ids = {len(input_ids)}")
for i in range(len(position_ids)):
    print(f"{position_ids[i]} : {input_ids[i]}")

len position ids = 31 , len input ids = 31
0 : 3838
1 : 374
2 : 279
3 : 6722
4 : 315
5 : 17689
6 : 30
7 : 14296
8 : 29
9 : 2208
10 : 8792
11 : 522
12 : 2047
13 : 29
7 : 14296
8 : 29
9 : 37036
10 : 1869
11 : 522
12 : 2047
13 : 29
7 : 14296
8 : 29
9 : 3428
10 : 22365
11 : 522
12 : 2047
13 : 29
7 : 16141
8 : 374
9 : 25


In [11]:
def get_attention_mask(idxs:dict) -> torch.tensor:
  dimension = idxs["answer_end_indx"] + 1
  attention_mask = torch.zeros((dimension,dimension))
  question_len = idxs['question_end_indx'] + 1
  attention_mask[:question_len,:question_len] = torch.tril(torch.ones(question_len,question_len))


  choice_start_idx = question_len
  for choice_end_idx in idxs["choices_end_indxs"]:
      attention_mask[choice_start_idx:choice_end_idx + 1,:question_len] = 1
      attention_mask[choice_start_idx:choice_end_idx + 1,choice_start_idx:choice_end_idx + 1] = torch.tril(torch.ones(choice_end_idx - choice_start_idx + 1,choice_end_idx - choice_start_idx + 1))
      choice_start_idx = choice_end_idx + 1

  answ_start_idx = idxs["choices_end_indxs"][-1] + 1

  attention_mask[answ_start_idx :,:answ_start_idx ] = 1
  attention_mask[answ_start_idx :,answ_start_idx :] = torch.tril(torch.ones(dimension - answ_start_idx ,dimension - answ_start_idx ))
  return attention_mask

In [12]:
attention_mask = get_attention_mask(idxs)
print(attention_mask.shape)
print("    " + "".join([f"{col:3d}" for col in range(attention_mask.shape[1])]))
print("    " + "_" * attention_mask.shape[1] * 3)
for i,row in enumerate(attention_mask):
    row_values = "".join([" x " if element == 1 else " - " for element in row])
    print(f"{i:2d} | {row_values}")


torch.Size([31, 31])
      0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30
    _____________________________________________________________________________________________
 0 |  x  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  - 
 1 |  x  x  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  - 
 2 |  x  x  x  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  - 
 3 |  x  x  x  x  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  - 
 4 |  x  x  x  x  x  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  - 
 5 |  x  x  x  x  x  x  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  - 
 6 |  x  x  x  x  x  x  x  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  - 
 7 |  x  x  x  x  x  x  x  x  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  

In [13]:
def forward(question : str, choices : list[str]):

  input_ids,idxs = create_input_tokens(question,choices)
  input_ids = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(device)
  position_ids = generate_position_ids(idxs)
  position_ids = torch.tensor(position_ids, dtype=torch.long).unsqueeze(0).to(device)
  attention_mask = get_attention_mask(idxs)
  attention_mask = attention_mask.to(dtype=torch.float32).to(device)
  attention_mask = torch.where(attention_mask == 1.0, 0.0, -float('inf'))
  attention_mask = attention_mask.unsqueeze(0).unsqueeze(0)

  output = model(
    input_ids = input_ids,
    attention_mask=attention_mask,
    position_ids=position_ids,
    past_key_values=None,
    inputs_embeds=None,
    use_cache=False,
  )

  choice_embeddings = torch.stack([output.last_hidden_state[0][choice_end_indx] for choice_end_indx in idxs["choices_end_indxs"]])
  answer_embedding = output.last_hidden_state[0][-1]
  return choice_embeddings,answer_embedding


In [14]:
question = "What is the capital of Spain?"
choices_1 = ["Valencia","Madrid","Barcelona"]
choices_2 = ["Valencia","Barcelona","Madrid"]

choice_embeddings_1,answer_embedding_1 = forward(question,choices_1)
choice_embeddings_2,answer_embedding_2 = forward(question,choices_2)





In [15]:

print("Valencia:", torch.allclose(choice_embeddings_1[0], choice_embeddings_2[0], atol=1e-4))
error_maximo = torch.max(torch.abs(choice_embeddings_1[0] - choice_embeddings_2[0])).item()
print(f"Error : {error_maximo}")

print("Madrid:", torch.allclose(choice_embeddings_1[1], choice_embeddings_2[2], atol=1e-4))
error_maximo = torch.max(torch.abs(choice_embeddings_1[1] - choice_embeddings_2[2])).item()
print(f"Error : {error_maximo}")

print("Barcelona:", torch.allclose(choice_embeddings_1[2], choice_embeddings_2[1], atol=1e-4))
error_maximo = torch.max(torch.abs(choice_embeddings_1[2] - choice_embeddings_2[1])).item()
print(f"Error : {error_maximo}")


print(f"Answer : {torch.allclose(answer_embedding_1, answer_embedding_2, atol=1e-4)}")

error_maximo = torch.max(torch.abs(answer_embedding_1 - answer_embedding_2)).item()
print(f"Error : {error_maximo}")

Valencia: True
Error : 0.0
Madrid: True
Error : 2.9087066650390625e-05
Barcelona: True
Error : 3.0279159545898438e-05
Answer : True
Error : 3.0517578125e-05


## Implementing Decision Head


In [16]:
import torch.nn as nn

class BilinearDecisionHead(nn.Module):
    def __init__(self, hidden_size):
      super().__init__()
      self.bilinear = nn.Bilinear(hidden_size, hidden_size, 1)

    def forward(self, answer_embedding, choice_embeddings):
      """
      answer_embedding: (batch_size, hidden_size)
      choice_embeddings: (batch_size, num_choices, hidden_size)
      """
      batch_size, num_choices, _ = choice_embeddings.shape


      answer_expanded = answer_embedding.unsqueeze(1).expand(-1, num_choices, -1)  # (batch_size,hidden_size) -> (batch_size, num_choices, hidden_size)

      scores = self.bilinear(answer_expanded, choice_embeddings) # Output: (batch_size, num_choices,1)
      scores = scores.squeeze(-1) # (batch_size, num_choices)

      return scores

In [17]:
hidden_size = model.config.hidden_size
decision_head = BilinearDecisionHead(hidden_size).to(device)

answer_batched = answer_embedding_1.unsqueeze(0)
choices_batched = choice_embeddings_1.unsqueeze(0)

logits = decision_head(answer_batched,choices_batched)
print("Logits: " , logits)
probabilities = torch.softmax(logits,dim=-1)
print("Probabilities: ", probabilities)
predicted_choice = torch.argmax(probabilities, dim=-1)
print("Predicted choice: ", predicted_choice)

Logits:  tensor([[-59.9619,  -2.7952, -40.8204]], device='cuda:0',
       grad_fn=<SqueezeBackward1>)
Probabilities:  tensor([[1.4888e-25, 1.0000e+00, 3.0612e-17]], device='cuda:0',
       grad_fn=<SoftmaxBackward0>)
Predicted choice:  tensor([1], device='cuda:0')


## Test Train
Let's train the model with a small portion of the dataset to check everything is working correctly. At this step we are only going to use the CHOICE entries of the dataset. Later we will add NOUL and SCORE


Dataset: https://huggingface.co/datasets/tasksource/tasksource-jev-typed-decisions

In [18]:
from datasets import load_dataset

dataset = load_dataset("tasksource/tasksource-jev-typed-decisions",split="train")

dataset = dataset.filter(lambda example: example["kind"] == "choice")

print(dataset[0:4])

README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

data/train-00000-of-00006.parquet: reconstructing file:   0%|          |  0.00B /  112MB            

data/train-00000-of-00006.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00006.parquet: reconstructing file:   0%|          |  0.00B /  112MB            

data/train-00001-of-00006.parquet: downloading bytes:           |  0.00B            

...

Generating train split:   0%|          | 0/999846 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/15000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/15000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/999846 [00:00<?, ? examples/s]

{'state': ['text_A: A $70 gift is expensive. A $30 gift is cheap.\ntext_B: A $26 gift is cheap.', "Where did the name crohn's disease come from?", "A: how do i have cyber sex?\nB: You can have cyber sex by using online messaging apps, webcams and other internet-based tools to communicate, send nude photos or videos, and even control each other's devices for sexual activities.", "First text:\nThe fears of Wilson and other government officials were confirmed when Galleanists—Italian immigrant followers of the anarchist Luigi Galleani—carried out a series of bombings in April and June 1919. At the end of April, some 30 Galleanist letter bombs had been mailed to a host of individuals, mostly prominent government officials and businessmen, but also law enforcement officials. Only a few reached their targets, but all exploded when opened. Some people suffered injuries, including a housekeeper in Senator Thomas W. Hardwick's residence, who had her hands blown off. On June 2, 1919, the second 

In [22]:
import torch.optim as optim

small_train_dataset = dataset.select(range(100))
num_epochs = 3

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(decision_head.parameters(), lr=1e-3)

model.eval()
for param in model.parameters():
    param.requires_grad = False

decision_head.train()
for param in decision_head.parameters():
    param.requires_grad = True

for epoch in range(num_epochs):
  total_loss = 0.0

  for batch in small_train_dataset:
    optimizer.zero_grad()

    text = batch["state"] + "\n" + batch["question"]
    choices = batch["options"]

    target_list = batch["target"]
    correct_index = target_list.index(1.0)

    choice_embeddings, answer_embeddings = forward(text,choices)

    answer_batched = answer_embeddings.unsqueeze(0)
    choices_batched = choice_embeddings.unsqueeze(0)

    logits = decision_head(answer_batched,choices_batched)

    target = torch.tensor([correct_index],dtype=torch.long).to(device)
    loss = criterion(logits,target)

    loss.backward()
    optimizer.step()

    total_loss += loss.item()

  average_loss = total_loss / len(small_train_dataset)
  print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {average_loss:.4f}")

Epoch 1/3, Loss: 937.6126
Epoch 2/3, Loss: 491.8370
Epoch 3/3, Loss: 436.2407
